In [ ]:
%pip install -q -r requirements.txt
%pip install -q "nbformat>=4.2.0"


In [1]:
from pathlib import Path
import sys

import hdbscan
import numpy as np
import pandas as pd
import plotly.express as px
import umap
from IPython.display import Markdown, display
from sklearn.metrics import silhouette_score

sys.path.insert(0, str(Path(".").resolve()))

CHALLENGES_CSV = Path("./output/processed/challenges.csv")
EXPECTATIONS_CSV = Path("./output/processed/expectations.csv")
PROCESSED_DIR = Path("./output/processed")
CHALLENGES_SCORED = PROCESSED_DIR / "challenges_scored.csv"
EXPECTATIONS_SCORED = PROCESSED_DIR / "expectations_scored.csv"
CATEGORIZED_CHALLENGES_CSV = PROCESSED_DIR / "categorized_challenges.csv"
CATEGORIZED_EXPECTATIONS_CSV = PROCESSED_DIR / "categorized_expectations.csv"
CATEGORY_SUMMARY_CSV = PROCESSED_DIR / "category_summary.csv"

CHALLENGE_TEXT_COL = "pain_points"
EXPECTATION_TEXT_COL = "expectations"

In [2]:
df_painpoints = pd.read_csv(CHALLENGES_CSV)
df_expectations = pd.read_csv(EXPECTATIONS_CSV)

if "source" not in df_painpoints.columns:
    raise ValueError("Missing `source`. Re-run: python3 setup/extract_records.py")

df_painpoints.loc[df_painpoints["focus_group"] == "NBD Kate", "focus_group"] = "NBD Internal"
df_expectations.loc[df_expectations["focus_group"] == "NBD Kate", "focus_group"] = "NBD Internal"

# Meeting notes only: merge ≤3-word lines into the next note (idempotent if extract already did it)
df_painpoints["source"] = df_painpoints["source"].replace({"notes": SOURCE_MEETING_NOTES})
df_expectations["source"] = df_expectations["source"].replace({"notes": SOURCE_MEETING_NOTES})

notes_mask = df_painpoints["source"] == SOURCE_MEETING_NOTES
df_painpoints = pd.concat(
    [
        df_painpoints.loc[~notes_mask],
        consolidate_short_with_next(
            df_painpoints.loc[notes_mask].copy(),
            text_col=CHALLENGE_TEXT_COL,
            max_words=NOTES_SHORT_MAX_WORDS,
        ),
    ],
    ignore_index=True,
)
exp_mask = df_expectations["source"] == SOURCE_MEETING_NOTES
df_expectations = pd.concat(
    [
        df_expectations.loc[~exp_mask],
        consolidate_short_with_next(
            df_expectations.loc[exp_mask].copy(),
            text_col=EXPECTATION_TEXT_COL,
            max_words=NOTES_SHORT_MAX_WORDS,
        ),
    ],
    ignore_index=True,
)

print(f"Challenges: {len(df_painpoints)} | Expectations: {len(df_expectations)}")
print(df_painpoints["source"].value_counts().to_string())


NameError: name 'SOURCE_MEETING_NOTES' is not defined